# 02 — Next-concept prediction with SONAR

**Finish notebook 01 first.** This notebook loads its exact context/next-sentence pairs and trained next-token model.
It then trains a small encoder–decoder Transformer to predict the embedding of the next sentence.

Here a **concept means one sentence represented by a fixed-size SONAR vector**.
The pretrained SONAR text encoder and decoder stay frozen. Only the small concept predictor is trained on this documentation.

~~~text
Previous sentences → frozen SONAR encoder → sentence vectors
                                                    ↓
                                 trainable encoder–decoder Transformer
                                                    ↓
                                      predicted next sentence vector
                                                    ↓
                                    frozen SONAR decoder → next sentence
~~~

This is an **LCM-inspired MSE baseline**, not a reproduction of the full Large Concept Model.
The [LCM paper](https://arxiv.org/abs/2412.08821) studies sentence-space prediction, including MSE and diffusion variants.
Its Base-LCM uses a decoder-only core; we retain your requested encoder–decoder architecture for this comparison.

Pretrained SONAR introduces outside knowledge. A result here cannot isolate the training objective from representation pretraining.


## 1. Prepare a compatible SONAR environment

Use a fresh **Python 3.10–3.12 Linux x86-64** kernel for the pinned Fairseq2 binary setup below.
SONAR depends on a matching PyTorch/Fairseq2/CUDA combination, so the setup pins a known published combination:
PyTorch 2.6.0, Fairseq2 0.5.2 and SONAR 0.5.0.

Fairseq2 also needs the system library libsndfile. On Ubuntu install it with `sudo apt-get install libsndfile1`; the setup cell installs it automatically in a root Colab runtime when absent.

The cell selects CUDA 12.4 wheels if an NVIDIA GPU is visible, otherwise CPU wheels.
Set `SKLEARNLM_SONAR_VARIANT` to `cpu` or `cu124` to choose explicitly.
If packages are changed, **restart the kernel and run from the top again** before importing binary modules.

The official text encoder and decoder checkpoints total about **7.6 GB** of downloads.
They load sequentially to limit memory use; a CUDA runtime is recommended.
Sources: [SONAR setup and API](https://github.com/facebookresearch/SONAR#installing), [Fairseq2 variants](https://github.com/facebookresearch/fairseq2#variants).


In [ ]:
import ctypes.util
import importlib.metadata
import importlib.util
import os
import platform
import shutil
import subprocess
import sys

SMOKE_TEST = os.environ.get("SKLEARNLM_SMOKE") == "1"
SMOKE_DEVICE = os.environ.get("SKLEARNLM_SMOKE_DEVICE", "cpu")
FIXTURE_TEST = os.environ.get("SKLEARNLM_CONCEPT_FIXTURE") == "1"
assert not FIXTURE_TEST or SMOKE_TEST, "Test fixtures are allowed only in explicit smoke mode."

if not FIXTURE_TEST:
    if not ((3, 10) <= sys.version_info[:2] <= (3, 12)
            and sys.platform == "linux" and platform.machine() == "x86_64"):
        raise RuntimeError("Use a Python 3.10–3.12 Linux x86-64 kernel for this pinned SONAR setup.")
    if not ctypes.util.find_library("sndfile"):
        if os.geteuid() == 0 and shutil.which("apt-get") and os.environ.get("SKLEARNLM_NO_INSTALL") != "1":
            subprocess.check_call(["apt-get", "update", "-qq"])
            subprocess.check_call(["apt-get", "install", "-y", "libsndfile1"])
        else:
            raise RuntimeError("SONAR needs the system libsndfile library. On Ubuntu: sudo apt-get install libsndfile1")

    def installed_version(name):
        try:
            return importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            return None

    expected = {"torch": "2.6.0", "torchaudio": "2.6.0", "fairseq2": "0.5.2",
                "fairseq2n": "0.5.2", "sonar-space": "0.5.0"}
    gpu_visible = False
    if shutil.which("nvidia-smi"):
        result = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                capture_output=True, text=True)
        gpu_visible = result.returncode == 0 and bool(result.stdout.strip())
    if not SMOKE_TEST and not gpu_visible:
        raise RuntimeError("A visible NVIDIA GPU is required. Select a GPU runtime or the local CUDA kernel.")
    variant = os.environ.get("SKLEARNLM_SONAR_VARIANT", "cu124" if gpu_visible else "cpu")
    assert variant in ("cpu", "cu124")
    needs_install = any((installed_version(name) or "").split("+")[0] != version
                        for name, version in expected.items())
    needs_install = needs_install or any(
        installed_version(name) != expected[name] + "+" + variant
        for name in ("torch", "torchaudio", "fairseq2n")
    )
    if needs_install:
        if os.environ.get("SKLEARNLM_NO_INSTALL") == "1":
            raise RuntimeError("Install the pinned SONAR environment described in notebooks/README.md.")
        pip = [sys.executable, "-m", "pip", "install"]
        subprocess.check_call(pip + ["torch==2.6.0", "torchaudio==2.6.0", "--index-url",
                                    f"https://download.pytorch.org/whl/{variant}"])
        subprocess.check_call(pip + ["fairseq2==0.5.2", f"fairseq2n==0.5.2+{variant}",
                                    "--extra-index-url",
                                    f"https://fair.pkg.atmeta.com/fairseq2/whl/pt2.6.0/{variant}"])
        subprocess.check_call(pip + ["sonar-space==0.5.0", "rouge-score>=0.1.2,<1",
                                    "tokenizers>=0.21,<1", "matplotlib>=3.8,<4"])
        raise RuntimeError("SONAR packages installed. Restart the kernel, then run all cells again.")

extra_requirements = {"rouge_score": "rouge-score>=0.1.2,<1",
                      "tokenizers": "tokenizers>=0.21,<1", "matplotlib": "matplotlib>=3.8,<4"}
missing = [package for module, package in extra_requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    if os.environ.get("SKLEARNLM_NO_INSTALL") == "1":
        raise RuntimeError(f"Missing dependencies: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import gc
import hashlib
import json
import math
from pathlib import Path
import random
import time
import zipfile

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from rouge_score import rouge_scorer
import sonar
from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline, EmbeddingToTextModelPipeline

assert bool(getattr(sonar, "__sklearnlm_test_fixture__", False)) == FIXTURE_TEST
print("SONAR backend:", "DETERMINISTIC TEST FIXTURE — NOT SONAR" if FIXTURE_TEST else "pretrained SONAR")
print("PyTorch:", torch.__version__)


## 2. Configuration and the first notebook's run

`WORD_RUN_DIR` must contain the exported files from notebook 01, including its selected inference weights and exact sentence pairs.
In Colab, extract `next_word_inference.zip` and point `SKLEARNLM_WORD_RUN_DIR` to that directory.

Defaults match notebook 01's width, attention heads, encoder/decoder depth, batch size, epochs and learning rate.
The vocabulary output layer is replaced by a continuous sentence-vector output, so parameter counts still differ.


In [ ]:
candidates = [Path.cwd() / "runs" / "next_word",
              Path.cwd().parent / "runs" / "next_word"]
WORD_RUN_DIR = Path(os.environ.get("SKLEARNLM_WORD_RUN_DIR",
                    str(next((p for p in candidates if (p / "experiment.json").exists()),
                             candidates[0]))))
RUN_DIR = Path(os.environ.get("SKLEARNLM_CONCEPT_RUN_DIR",
                             str(WORD_RUN_DIR.parent / "next_concept")))
RUN_DIR.mkdir(parents=True, exist_ok=True)
required = ["experiment.json", "inference_model.pt", "sentence_pairs.jsonl",
            "pair_manifest.json", "tokenizer.json", "metrics.json", "history.json"]
assert all((WORD_RUN_DIR / name).is_file() for name in required), \
    "Finish notebook 01, or extract its next_word_inference.zip and set WORD_RUN_DIR."

word_experiment = json.loads((WORD_RUN_DIR / "experiment.json").read_text())
assert word_experiment["architecture"] == "encoder_decoder"
assert word_experiment["objective"] == "next_token"
assert word_experiment["data_mode"] == "sentences", "Use notebook 01's sentence mode for this comparison."
assert word_experiment["smoke_test"] == SMOKE_TEST, "Do not mix smoke artifacts with real runs."

WCFG = word_experiment["config"]
CFG = {key: WCFG[key] for key in ("d_model", "n_heads", "encoder_layers",
       "decoder_layers", "dropout", "batch_size", "epochs", "learning_rate",
       "weight_decay", "patience", "max_steps", "context_sentences")}
SEED = word_experiment["seed"]
RUN_TRAINING = True
RUN_TEST_EVALUATION = SMOKE_TEST
COMPARISON_SPLIT = "validation"
COMPARISON_EXAMPLES = 2 if SMOKE_TEST else 32
COMPARISON_SEED = 1729
SONAR_BATCH_SIZE = 2  # Conservative default for a 6 GB laptop GPU.
SONAR_MAX_TOKENS = 512
SONAR_DECODE_LIMIT = 128
ENCODER_CARD = "text_sonar_basic_encoder"
DECODER_CARD = "text_sonar_basic_decoder"
LANGUAGE = "eng_Latn"

DEVICE = torch.device(SMOKE_DEVICE if SMOKE_TEST else "cuda")
if DEVICE.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU required. Select the SKlearnLM CUDA kernel and install the CUDA requirements, "
        "or enable a GPU runtime in Colab. This notebook will not silently train on CPU."
    )
assert DEVICE.type in ("cpu", "cuda")
SONAR_DEVICE = DEVICE
SONAR_DTYPE = torch.float16 if SONAR_DEVICE.type == "cuda" else torch.float32
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if SMOKE_TEST:
    torch.set_num_threads(2)
print("Word run:", WORD_RUN_DIR.resolve())
print("Concept run:", RUN_DIR.resolve())
print("Device:", DEVICE)
print(json.dumps(CFG, indent=2))


## GPU training and mixed precision

Normal training requires a CUDA GPU and stops with a clear message if one is unavailable.
Select the project's CUDA Jupyter kernel, or choose a GPU runtime in Colab.

Automatic mixed precision uses bfloat16 when the GPU supports it, otherwise float16 with gradient scaling.
Model weights stay in float32. Forward/loss computation uses autocast; backward runs outside that context.
Gradients are unscaled before clipping. Checkpoints retain the scaler state.

The CPU path is reserved for explicit automated smoke tests. A CUDA smoke test uses the normal model dimensions
and performs two actual optimizer updates. Device, precision and peak allocated GPU memory are recorded.
See [PyTorch AMP examples](https://docs.pytorch.org/docs/2.6/notes/amp_examples.html).


In [ ]:
USE_AMP = DEVICE.type == "cuda"
precision_choice = os.environ.get("SKLEARNLM_PRECISION", "auto")
assert precision_choice in ("auto", "bf16", "fp16", "fp32")
if precision_choice == "bf16" and USE_AMP and not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support bfloat16; use fp16 or auto.")
AMP_DTYPE = (torch.bfloat16 if USE_AMP and
             (precision_choice == "bf16" or
              (precision_choice == "auto" and torch.cuda.is_bf16_supported()))
             else torch.float16)
USE_AMP = USE_AMP and precision_choice != "fp32"

def amp_context():
    return torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP)

RUNTIME = {
    "device": DEVICE.type,
    "torch_version": str(torch.__version__),
    "cuda_runtime": torch.version.cuda,
    "mixed_precision": USE_AMP,
    "training_precision": str(AMP_DTYPE) if USE_AMP else "torch.float32",
}
if DEVICE.type == "cuda":
    RUNTIME["gpu_name"] = torch.cuda.get_device_name(DEVICE)
    RUNTIME["total_vram_bytes"] = torch.cuda.get_device_properties(DEVICE).total_memory
    with amp_context():
        probe = torch.ones(32, 32, device=DEVICE) @ torch.ones(32, 32, device=DEVICE)
    assert torch.isfinite(probe).all()
    if USE_AMP:
        assert probe.dtype == AMP_DTYPE
    torch.cuda.synchronize()
    del probe
print(json.dumps(RUNTIME, indent=2))


## 3. Reuse the exact same examples

No new split or sentence segmentation is performed. The first notebook's pair fingerprint must match.
The context consists only of earlier sentences; the target sentence is never passed to the predictor.

We may **encode** held-out text with a frozen pretrained encoder, but normalization and neural training use training units only.


In [ ]:
def sha256(payload):
    return hashlib.sha256(payload).hexdigest()

pair_payload = (WORD_RUN_DIR / "sentence_pairs.jsonl").read_bytes()
PAIRS_SHA = sha256(pair_payload)
assert PAIRS_SHA == word_experiment["pairs_sha256"]
pair_manifest = json.loads((WORD_RUN_DIR / "pair_manifest.json").read_text())
assert PAIRS_SHA == pair_manifest["sha256"]
pairs = {split: [] for split in ("train", "validation", "test")}
for line in pair_payload.decode("utf-8").splitlines():
    row = json.loads(line)
    pairs[row["split"]].append(row)
assert {s: len(rows) for s, rows in pairs.items()} == pair_manifest["counts"]
assert len({row["id"] for rows in pairs.values() for row in rows}) == sum(map(len, pairs.values()))
doc_sets = {s: {row["document_id"] for row in rows} for s, rows in pairs.items()}
assert not (doc_sets["train"] & doc_sets["validation"] or doc_sets["train"] & doc_sets["test"]
            or doc_sets["validation"] & doc_sets["test"])

sentences = sorted({text for rows in pairs.values() for row in rows
                    for text in row["context"] + [row["target"]]})
sentence_index = {text: index for index, text in enumerate(sentences)}
train_sentences = sorted({text for row in pairs["train"] for text in row["context"] + [row["target"]]})
print("Exact pair counts:", pair_manifest["counts"])
print("Unique sentence units:", len(sentences))
print("Example context:", pairs["train"][0]["context"])
print("Target sentence:", pairs["train"][0]["target"])


## 4. Load frozen SONAR pipelines lazily

SONAR maps each sentence to a 1,024-dimensional vector and can decode a vector back to text.
The helper functions below keep only one large SONAR pipeline loaded at a time.

Both pipelines are put in evaluation mode and frozen. No gradients or optimizer updates reach their weights.
Their parameter counts and package/model-card provenance are recorded separately from the trainable concept predictor.
The weights are downloaded by SONAR from the official asset URLs; they are not bundled in this repository.


In [ ]:
embedder, text_decoder = None, None
frozen_parameters = {}
model_load_seconds = {"encoder": 0.0, "decoder": 0.0}

def clear_sonar():
    global embedder, text_decoder
    embedder, text_decoder = None, None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def get_embedder():
    global embedder
    if embedder is None:
        clear_sonar()
        started = time.perf_counter()
        embedder = TextToEmbeddingModelPipeline(
            encoder=ENCODER_CARD, tokenizer=ENCODER_CARD,
            device=SONAR_DEVICE, dtype=SONAR_DTYPE,
        ).eval()
        embedder.requires_grad_(False)
        frozen_parameters["encoder"] = sum(p.numel() for p in embedder.parameters())
        model_load_seconds["encoder"] += time.perf_counter() - started
    return embedder

def get_text_decoder():
    global text_decoder
    if text_decoder is None:
        clear_sonar()
        started = time.perf_counter()
        text_decoder = EmbeddingToTextModelPipeline(
            decoder=DECODER_CARD, tokenizer=ENCODER_CARD,
            device=SONAR_DEVICE, dtype=SONAR_DTYPE,
        ).eval()
        text_decoder.requires_grad_(False)
        frozen_parameters["decoder"] = sum(p.numel() for p in text_decoder.parameters())
        model_load_seconds["decoder"] += time.perf_counter() - started
    return text_decoder

def encode_texts(texts):
    pipeline = get_embedder()
    # Reject overlong units instead of accepting the pipeline's truncation.
    encoder = pipeline.tokenizer.create_encoder(lang=LANGUAGE, device=torch.device("cpu"))
    assert all(encoder(text).numel() <= SONAR_MAX_TOKENS for text in texts), \
        "A sentence exceeds SONAR's configured limit; revise the shared dataset before comparing."
    vectors = pipeline.predict(texts, source_lang=LANGUAGE, batch_size=SONAR_BATCH_SIZE,
                               max_seq_len=SONAR_MAX_TOKENS)
    return vectors.detach().cpu().float().clone()

def decode_vectors(vectors):
    pipeline = get_text_decoder()
    return pipeline.predict(vectors.to(device=SONAR_DEVICE, dtype=SONAR_DTYPE),
                            target_lang=LANGUAGE, batch_size=4, beam_size=1,
                            max_seq_len=SONAR_DECODE_LIMIT)

def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


## 5. Encode sentences and cache their vectors

The expensive SONAR encoding is done once per unique sentence. A cache must match the exact sentence list, backend, model cards and precision.
The separate fixture backend is used only by CI to exercise notebook wiring without downloading large models.
Fixture outputs are always marked as such and cannot be used as research results.

Model-card digests identify the installed asset definitions; upstream checkpoint URLs are recorded.
Checkpoint binary hashes are not supplied by those cards, so retain the downloaded cache when reproducing an exact experiment.


In [ ]:
from importlib.resources import files as package_files

if FIXTURE_TEST:
    sonar_provenance = {"backend": "test_fixture_not_sonar", "packages": {}}
else:
    versions = {name: importlib.metadata.version(name)
                for name in ("sonar-space", "fairseq2", "fairseq2n", "torch", "torchaudio")}
    cards = {name: sha256(package_files("sonar.cards").joinpath(name + ".yaml").read_bytes())
             for name in (ENCODER_CARD, DECODER_CARD)}
    sonar_provenance = {
        "backend": "pretrained_sonar", "packages": versions, "model_card_sha256": cards,
        "encoder_checkpoint_url": "https://dl.fbaipublicfiles.com/SONAR/sonar_text_encoder.pt",
        "decoder_checkpoint_url": "https://dl.fbaipublicfiles.com/SONAR/sonar_text_decoder.pt",
        "license_url": "https://github.com/facebookresearch/SONAR/blob/main/NC_MODEL_LICENSE.md",
    }

embedding_info = {
    "pairs_sha256": PAIRS_SHA,
    "sentences_sha256": sha256(json.dumps(sentences, ensure_ascii=False).encode()),
    "encoder_card": ENCODER_CARD, "decoder_card": DECODER_CARD, "language": LANGUAGE,
    "dtype": str(SONAR_DTYPE), "max_tokens": SONAR_MAX_TOKENS,
    "provenance": sonar_provenance, "fixture_test": FIXTURE_TEST,
}
embedding_path = RUN_DIR / "sentence_embeddings.pt"
embedding_seconds_this_run = 0.0
if embedding_path.exists():
    cached = torch.load(embedding_path, map_location="cpu", weights_only=True)
    assert cached["info"] == embedding_info, "Choose a fresh concept RUN_DIR."
    embeddings = cached["embeddings"]
    embedding_seconds_original = cached["encoding_seconds"]
    frozen_parameters.update(cached["frozen_parameters"])
else:
    synchronize()
    started = time.perf_counter()
    chunks = []
    for start in range(0, len(sentences), 256):
        chunks.append(encode_texts(sentences[start:start + 256]))
        print("Encoded", min(start + 256, len(sentences)), "/", len(sentences))
    embeddings = torch.cat(chunks)
    synchronize()
    embedding_seconds_this_run = time.perf_counter() - started
    embedding_seconds_original = embedding_seconds_this_run
    torch.save({"info": embedding_info, "embeddings": embeddings,
                "encoding_seconds": embedding_seconds_original,
                "frozen_parameters": frozen_parameters}, embedding_path)
assert embeddings.shape[0] == len(sentences) and torch.isfinite(embeddings).all()
assert embeddings.shape[1] == 1024
CFG["concept_dim"] = embeddings.shape[1]
clear_sonar()
print("Sentence matrix:", embeddings.shape)
print("Encoding seconds this run:", embedding_seconds_this_run)


## 6. Fit normalization on training sentences only

SONAR dimensions have different scales. We subtract each training median and divide by its interquartile range.
This follows the paper's robust-scaling idea. A small floor avoids division by zero.

The scaler is fit on unique sentences that occur in **training pairs only**.
Validation/test vectors never contribute to these statistics.
The predicted standardized vector is converted back to the original SONAR space before text decoding.


In [ ]:
normalization_started = time.perf_counter()
train_indices = [sentence_index[text] for text in train_sentences]
training_vectors = embeddings[train_indices]
center = training_vectors.median(dim=0).values
scale = (torch.quantile(training_vectors, 0.75, dim=0)
         - torch.quantile(training_vectors, 0.25, dim=0)).clamp_min(1e-3)
normalized = (embeddings - center) / scale
assert torch.allclose(normalized * scale + center, embeddings, atol=1e-5)
normalization_info = {"fit_split": "train", "unique_training_sentences": len(train_sentences),
                      "training_sentence_sha256": sha256(json.dumps(train_sentences).encode()),
                      "method": "median_and_iqr", "scale_floor": 1e-3}
torch.save({"center": center, "scale": scale, "info": normalization_info},
           RUN_DIR / "normalization.pt")
del training_vectors
print(normalization_info)

NORMALIZATION_SECONDS = time.perf_counter() - normalization_started


## 7. Dataset: earlier sentence vectors → next sentence vector

Each source has one or two sentence vectors. We pad short contexts and keep a Boolean validity mask.
There is one target vector per example.

This next-concept predictor uses one learned start query in the decoder. No target embedding is supplied as decoder input.
Its one-position decoder self-attention mask is therefore trivial; cross-attention reads the encoded context.
For a longer rollout, we append the predicted concept to the context and predict another one.


In [ ]:
class ConceptDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        source = normalized[[sentence_index[text] for text in row["context"]]]
        target = normalized[sentence_index[row["target"]]]
        return source, target

def collate_concepts(batch):
    source, target = zip(*batch)
    lengths = torch.tensor([len(x) for x in source])
    source = pad_sequence(source, batch_first=True, padding_value=0.0)
    valid = torch.arange(source.size(1))[None, :] < lengths[:, None]
    return source, valid, torch.stack(target)

loaders = {}
for split, rows in pairs.items():
    loaders[split] = DataLoader(
        ConceptDataset(rows), batch_size=CFG["batch_size"], shuffle=(split == "train"),
        collate_fn=collate_concepts, num_workers=0, pin_memory=(DEVICE.type == "cuda"),
        generator=torch.Generator().manual_seed(SEED),
    )
source, valid, target = next(iter(loaders["train"]))
print("Context vectors:", source.shape, "| valid mask:", valid.shape, "| targets:", target.shape)


## 8. The same simple Transformer building blocks

These are the same explicit attention, feed-forward, encoder and decoder classes used in notebook 01.
The inputs are sentence vectors instead of token embeddings. Attention works over **sentence positions**.

Keeping these layers visible makes the code readable without importing a hidden model implementation.


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_length, dropout):
        super().__init__()
        assert d_model % 2 == 0
        position = torch.arange(max_length).float().unsqueeze(1)
        frequency = torch.exp(torch.arange(0, d_model, 2).float()
                              * (-math.log(10000.0) / d_model))
        values = torch.zeros(max_length, d_model)
        values[:, 0::2] = torch.sin(position * frequency)
        values[:, 1::2] = torch.cos(position * frequency)
        self.register_buffer("values", values.unsqueeze(0))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(x + self.values[:, :x.size(1)])


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        self.output = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        batch, length, width = x.shape
        return x.reshape(batch, length, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        q = self.split_heads(self.query(query))
        k = self.split_heads(self.key(key))
        v = self.split_heads(self.value(value))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        attended = self.dropout(weights) @ v
        batch, heads, length, head_dim = attended.shape
        attended = attended.transpose(1, 2).contiguous().reshape(batch, length, heads * head_dim)
        return self.output(attended), weights


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, x):
        return self.layers(x)


In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.feed_forward = FeedForward(d_model, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, source_mask):
        attended, _ = self.attention(x, x, x, source_mask)
        x = self.norm1(x + self.dropout(attended))
        x = self.norm2(x + self.dropout(self.feed_forward(x)))
        return x


In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.feed_forward = FeedForward(d_model, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, source_mask, decoder_mask):
        attended, _ = self.self_attention(x, x, x, decoder_mask)
        x = self.norm1(x + self.dropout(attended))
        attended, _ = self.cross_attention(x, memory, memory, source_mask)
        x = self.norm2(x + self.dropout(attended))
        x = self.norm3(x + self.dropout(self.feed_forward(x)))
        return x


## 9. Assemble the concept encoder–decoder

An input linear layer projects a standardized SONAR vector to the Transformer's hidden width.
The encoder processes the previous concepts.
A learned start vector queries that memory through the decoder.
The final linear layer predicts all sentence-vector dimensions together.

The predictor starts from random weights. SONAR's pretrained encoder and decoder are separate frozen components.


In [ ]:
class ConceptTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        width = config["d_model"]
        self.input_projection = nn.Linear(config["concept_dim"], width)
        self.positions = PositionalEncoding(width, config["context_sentences"] + 2,
                                             config["dropout"])
        self.start = nn.Parameter(torch.zeros(1, 1, width))
        self.encoder = nn.ModuleList([
            EncoderBlock(width, config["n_heads"], config["dropout"])
            for _ in range(config["encoder_layers"])
        ])
        self.decoder = nn.ModuleList([
            DecoderBlock(width, config["n_heads"], config["dropout"])
            for _ in range(config["decoder_layers"])
        ])
        self.output = nn.Linear(width, config["concept_dim"])

    def forward(self, source, valid):
        source_mask = valid[:, None, None, :]
        memory = self.positions(self.input_projection(source))
        for block in self.encoder:
            memory = block(memory, source_mask)
        query = self.positions(self.start.expand(source.size(0), 1, -1))
        decoder_mask = torch.ones(1, 1, 1, 1, dtype=torch.bool, device=source.device)
        for block in self.decoder:
            query = block(query, memory, source_mask, decoder_mask)
        return self.output(query[:, 0])

model = ConceptTransformer(CFG).to(DEVICE)
PARAMETERS = sum(p.numel() for p in model.parameters())
print(model)
print("Trainable predictor parameters:", PARAMETERS)
print("Frozen SONAR parameters loaded so far:", frozen_parameters)


## 10. Check shapes, padding and gradients

Changing a masked-out context vector must not change the prediction.
The decoder has no access to the true target, so target leakage is avoided by construction.


In [ ]:
source, valid, target = [x.to(DEVICE) for x in next(iter(loaders["train"]))]
model.eval()
with torch.no_grad():
    prediction = model(source, valid)
    assert prediction.shape == target.shape
    padded = F.pad(source[:1], (0, 0, 0, 1), value=123.0)
    padded_valid = F.pad(valid[:1], (0, 1), value=False)
    assert torch.allclose(prediction[:1], model(padded, padded_valid), atol=1e-5)
    changed = source[:1].clone()
    changed[:, 0] += 1.0
    assert not torch.allclose(prediction[:1], model(changed, valid[:1]), atol=1e-6)
loss = F.mse_loss(model(source, valid), target)
loss.backward()
assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
model.zero_grad(set_to_none=True)
checks = {"shape": True, "padding_invariance": True, "uses_context": True,
          "finite_gradients": True, "target_not_an_input": True}
if USE_AMP:
    with torch.no_grad(), amp_context():
        amp_prediction = model(source, valid)
    assert amp_prediction.dtype == AMP_DTYPE and torch.isfinite(amp_prediction).all()
    checks["mixed_precision_forward"] = True
(RUN_DIR / "model_checks.json").write_text(json.dumps(checks, indent=2) + "\n")
print(checks)


## 11. MSE, cosine similarity and simple baselines

Training minimizes mean squared error in standardized sentence-vector space.
Cosine similarity is measured after restoring the original SONAR scale.

Two simple baselines help interpret the result: always predict the training center, or copy the last context sentence vector.
The paper reports limitations of simple MSE concept prediction; lower MSE alone does not establish good text generation.


In [ ]:
@torch.no_grad()
@torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP)
def evaluate(model, loader, max_batches=None):
    was_training = model.training
    model.eval()
    total_mse, total_cosine, center_cosine, copy_cosine, examples = 0.0, 0.0, 0.0, 0.0, 0
    for index, batch in enumerate(loader):
        if max_batches is not None and index >= max_batches:
            break
        source, valid, target = [x.to(DEVICE, non_blocking=True) for x in batch]
        prediction = model(source, valid)
        count = len(target)
        total_mse += F.mse_loss(prediction, target, reduction="none").mean(-1).sum().item()
        raw_target = target.cpu() * scale + center
        raw_prediction = prediction.float().cpu() * scale + center
        last = source[torch.arange(count, device=DEVICE), valid.sum(1) - 1]
        raw_last = last.cpu() * scale + center
        total_cosine += F.cosine_similarity(raw_prediction, raw_target, dim=-1).sum().item()
        center_cosine += F.cosine_similarity(center.expand_as(raw_target), raw_target, dim=-1).sum().item()
        copy_cosine += F.cosine_similarity(raw_last, raw_target, dim=-1).sum().item()
        examples += count
    model.train(was_training)
    assert examples > 0
    return {"standardized_mse": total_mse / examples, "raw_sonar_cosine": total_cosine / examples,
            "training_center_cosine": center_cosine / examples, "copy_last_cosine": copy_cosine / examples,
            "examples": examples, "scope": "two_batches_only" if max_batches == 2 else "full_split"}


## 12. Train the next-concept predictor

The optimizer updates only the small concept Transformer. SONAR is already frozen and its vectors are cached.
The learning rate, batch size and epoch budget match notebook 01.
Validation MSE chooses the checkpoint; test data never chooses it.

Wall time below includes training and validation on cached embeddings. The separate preprocessing and pretrained loading costs are retained.
Equal epochs or optimizer updates do not mean equal FLOPs or equal model capacity.


In [ ]:
experiment = {
    "objective": "next_concept_mse", "architecture": "encoder_decoder",
    "config": CFG, "seed": SEED, "pairs_sha256": PAIRS_SHA,
    "word_experiment": word_experiment, "embedding_info": embedding_info,
    "normalization": normalization_info, "smoke_test": SMOKE_TEST,
    "fixture_test": FIXTURE_TEST, "training_precision": RUNTIME["training_precision"],
}
experiment_path = RUN_DIR / "experiment.json"
if experiment_path.exists():
    assert json.loads(experiment_path.read_text()) == experiment, "Use a fresh concept RUN_DIR."
else:
    experiment_path.write_text(json.dumps(experiment, indent=2) + "\n")

def save_checkpoint(path, optimizer, scaler, step, history):
    torch.save({"model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
                "experiment": experiment, "center": center, "scale": scale,
                "scaler_state": scaler.state_dict(), "runtime": RUNTIME,
                "step": step, "history": history}, path)

def train():
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["learning_rate"],
                                 weight_decay=CFG["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
    history, step, best, stale = [], 0, float("inf"), 0
    skipped_updates = 0
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    for epoch in range(1, CFG["epochs"] + 1):
        model.train()
        total_loss, count = 0.0, 0
        for batch in loaders["train"]:
            source, valid, target = [x.to(DEVICE, non_blocking=True) for x in batch]
            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                loss = F.mse_loss(model(source, valid), target)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite concept training loss.")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0,
                                     error_if_nonfinite=not scaler.is_enabled())
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() < old_scale:
                skipped_updates += 1
                continue
            total_loss += loss.item() * len(target)
            count += len(target)
            step += 1
            if step % 100 == 0:
                print(f"epoch {epoch}, step {step}: batch MSE {loss.item():.4f}")
            if CFG["max_steps"] is not None and step >= CFG["max_steps"]:
                break
        assert count > 0, "No finite optimizer updates; try bfloat16 or float32."
        validation = evaluate(model, loaders["validation"], 2 if SMOKE_TEST else None)
        row = {"epoch": epoch, "step": step, "train_loss": total_loss / count,
               "validation_loss": validation["standardized_mse"],
               "validation_cosine": validation["raw_sonar_cosine"],
               "elapsed_seconds": time.perf_counter() - started,
               "amp_skipped_updates": skipped_updates}
        history.append(row)
        print(row)
        save_checkpoint(RUN_DIR / "last.pt", optimizer, scaler, step, history)
        if row["validation_loss"] < best:
            best, stale = row["validation_loss"], 0
            save_checkpoint(RUN_DIR / "best.pt", optimizer, scaler, step, history)
        else:
            stale += 1
        (RUN_DIR / "history.json").write_text(json.dumps(history, indent=2) + "\n")
        if stale >= CFG["patience"] or (CFG["max_steps"] is not None and step >= CFG["max_steps"]):
            break
    return history

history = train() if RUN_TRAINING else []


## 13. Restore and evaluate

All reported metrics come from actual execution.
A normal run evaluates the full validation split; smoke mode uses two batches.
Final test evaluation remains separate and disabled until model selection is finished.


In [ ]:
best_path = RUN_DIR / "best.pt"
assert best_path.exists(), "Train the concept model before evaluating."
checkpoint = torch.load(best_path, map_location=DEVICE, weights_only=True)
assert checkpoint["experiment"] == experiment
model.load_state_dict(checkpoint["model_state"])
model.eval()
restored = ConceptTransformer(CFG).to(DEVICE)
restored.load_state_dict(checkpoint["model_state"])
restored.eval()
with torch.no_grad():
    assert torch.allclose(model(source, valid), restored(source, valid), atol=1e-6)
del restored
metrics = {
    "objective": "next_concept_mse", "architecture": "encoder_decoder",
    "smoke_test": SMOKE_TEST, "fixture_test": FIXTURE_TEST,
    "live_sonar_used": not FIXTURE_TEST, "parameters": PARAMETERS, "runtime": RUNTIME,
    "peak_cuda_bytes": torch.cuda.max_memory_allocated() if DEVICE.type == "cuda" else None,
    "optimizer_updates": history[-1]["step"] if history else checkpoint["step"],
    "selected_checkpoint_step": checkpoint["step"],
    "training_seconds": history[-1]["elapsed_seconds"] if history else None,
    "embedding_seconds_this_run": embedding_seconds_this_run,
    "embedding_seconds_original": embedding_seconds_original,
    "normalization_seconds": NORMALIZATION_SECONDS,
    "validation": evaluate(model, loaders["validation"], 2 if SMOKE_TEST else None),
    "full_training_performed": not SMOKE_TEST and bool(history),
}
print(json.dumps(metrics["validation"], indent=2))


In [ ]:
if RUN_TEST_EVALUATION:
    metrics["test"] = evaluate(model, loaders["test"], 2 if SMOKE_TEST else None)
    print(json.dumps(metrics["test"], indent=2))
else:
    print("Final test evaluation disabled: select the experiment with validation first.")


## 14. Compare measured training curves

Token cross-entropy and concept MSE use different scales and objectives.
They appear in separate panels. Their absolute numeric values cannot establish which model is better.


In [ ]:
word_history = json.loads((WORD_RUN_DIR / "history.json").read_text())
if history and word_history:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, rows, title, ylabel in [
        (axes[0], word_history, "Next token", "Cross-entropy per token"),
        (axes[1], history, "Next concept", "Standardized vector MSE"),
    ]:
        ax.plot([row["epoch"] for row in rows], [row["train_loss"] for row in rows], "o-", label="Train")
        ax.plot([row["epoch"] for row in rows], [row["validation_loss"] for row in rows], "o-", label="Validation")
        ax.set(xlabel="Epoch", ylabel=ylabel, title=title)
        ax.legend()
    fig.tight_layout()
    fig.savefig(RUN_DIR / "training_comparison.png")
    plt.show()


## 15. Reload the next-token baseline for a common text evaluation

The code below is the first notebook's model definition and greedy decoding logic.
The saved tokenizer, special IDs, configuration and state dictionary must match its experiment fingerprint.

Neither model sees the reference target during free generation.


In [ ]:
inference = torch.load(WORD_RUN_DIR / "inference_model.pt", map_location="cpu", weights_only=True)
assert inference["experiment"] == word_experiment
PAD_ID, BOS_ID, EOS_ID = inference["pad_id"], inference["bos_id"], inference["eos_id"]
tokenizer = Tokenizer.from_file(str(WORD_RUN_DIR / "tokenizer.json"))
assert sha256((WORD_RUN_DIR / "tokenizer.json").read_bytes()) == word_experiment["tokenizer_sha256"]

def make_masks(source, decoder_input):
    source_mask = (source != PAD_ID)[:, None, None, :]
    length = decoder_input.size(1)
    causal = torch.tril(torch.ones(length, length, dtype=torch.bool, device=source.device))
    decoder_mask = (decoder_input != PAD_ID)[:, None, None, :] & causal[None, None, :, :]
    return source_mask, decoder_mask


In [ ]:
class WordTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        width = config["d_model"]
        self.scale = math.sqrt(width)
        self.embedding = nn.Embedding(config["vocab_size"], width, padding_idx=PAD_ID)
        max_length = max(config["source_length"], config["target_length"]) + 1
        self.positions = PositionalEncoding(width, max_length, config["dropout"])
        self.encoder = nn.ModuleList([
            EncoderBlock(width, config["n_heads"], config["dropout"])
            for _ in range(config["encoder_layers"])
        ])
        self.decoder = nn.ModuleList([
            DecoderBlock(width, config["n_heads"], config["dropout"])
            for _ in range(config["decoder_layers"])
        ])
        self.output = nn.Linear(width, config["vocab_size"])

    def encode(self, source, source_mask):
        memory = self.positions(self.embedding(source) * self.scale)
        for block in self.encoder:
            memory = block(memory, source_mask)
        return memory

    def decode(self, decoder_input, memory, source_mask, decoder_mask):
        x = self.positions(self.embedding(decoder_input) * self.scale)
        for block in self.decoder:
            x = block(x, memory, source_mask, decoder_mask)
        return self.output(x)

    def forward(self, source, decoder_input):
        source_mask, decoder_mask = make_masks(source, decoder_input)
        memory = self.encode(source, source_mask)
        return self.decode(decoder_input, memory, source_mask, decoder_mask)


In [ ]:
word_model = WordTransformer(WCFG).to(DEVICE)
word_model.load_state_dict(inference["model_state"])
word_model.eval()
del inference

@torch.no_grad()
@torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP)
def generate_word(source_text):
    source = torch.tensor([tokenizer.encode(source_text).ids], device=DEVICE)
    assert 0 < source.size(1) <= WCFG["source_length"]
    source_mask = (source != PAD_ID)[:, None, None, :]
    memory = word_model.encode(source, source_mask)
    generated = torch.tensor([[BOS_ID]], device=DEVICE)
    for _ in range(12 if SMOKE_TEST else WCFG["target_length"]):
        _, decoder_mask = make_masks(source, generated)
        logits = word_model.decode(generated, memory, source_mask, decoder_mask)[:, -1]
        logits[:, [PAD_ID, BOS_ID]] = float("-inf")
        next_token = logits.argmax(-1, keepdim=True)
        generated = torch.cat((generated, next_token), dim=1)
        if next_token.item() == EOS_ID:
            break
    return tokenizer.decode(generated[0, 1:].tolist(), skip_special_tokens=True)


## 16. Generate both continuations on a fixed held-out subset

We select a deterministic subset of validation pairs, independent of the training seed.
Generation is greedy in both text decoders, with separately recorded token limits because their tokenizers differ.

We also decode the **true target embedding**. That is a reconstruction diagnostic showing the SONAR bottleneck;
it is labelled an oracle reference reconstruction and is never used as a model prediction.

Cached context vectors are used for concept generation here. Timings exclude context embedding and model loading,
so these numbers are **not end-to-end speed comparisons**.


In [ ]:
assert COMPARISON_SPLIT in ("validation", "test")
assert COMPARISON_SPLIT != "test" or RUN_TEST_EVALUATION, "Enable final test evaluation explicitly."
indices = list(range(len(pairs[COMPARISON_SPLIT])))
random.Random(COMPARISON_SEED).shuffle(indices)
chosen = [pairs[COMPARISON_SPLIT][i] for i in indices[:COMPARISON_EXAMPLES]]

word_outputs = []
synchronize()
started = time.perf_counter()
for row in chosen:
    word_outputs.append(generate_word(row["source"]))
synchronize()
word_generation_seconds = time.perf_counter() - started

raw_predictions, raw_targets = [], []
synchronize()
started = time.perf_counter()
with torch.no_grad():
    for row in chosen:
        x = normalized[[sentence_index[text] for text in row["context"]]].unsqueeze(0).to(DEVICE)
        valid_context = torch.ones(1, x.size(1), dtype=torch.bool, device=DEVICE)
        with amp_context():
            prediction = model(x, valid_context).float().cpu()[0]
        raw_predictions.append(prediction * scale + center)
        raw_targets.append(embeddings[sentence_index[row["target"]]])
synchronize()
concept_predictor_seconds = time.perf_counter() - started

get_text_decoder()  # Loading time is recorded separately.
synchronize()
started = time.perf_counter()
concept_outputs = decode_vectors(torch.stack(raw_predictions))
synchronize()
concept_decode_seconds = time.perf_counter() - started
oracle_reconstructions = decode_vectors(torch.stack(raw_targets))
clear_sonar()
print("Generated", len(chosen), "continuations per model.")


## 17. Score the same generated text

ROUGE-L F1 measures word-sequence overlap with the reference sentence.
SONAR cosine measures similarity between a generated sentence embedding and its reference embedding.
Empty generations receive a semantic score of zero.

SONAR cosine favors the representation used by the concept model, so interpret it alongside ROUGE-L and human inspection.
Neither metric proves technical accuracy. The table below is a small held-out subset, not a full-split generation benchmark.


In [ ]:
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
generated_texts = word_outputs + concept_outputs + oracle_reconstructions
generated_vectors = encode_texts([text if text.strip() else "." for text in generated_texts])
clear_sonar()
rows = []
size = len(chosen)
for index, row in enumerate(chosen):
    result = {"pair_id": row["id"], "split": COMPARISON_SPLIT, "context": row["source"],
              "reference": row["target"], "oracle_sonar_reconstruction": oracle_reconstructions[index]}
    reference_vector = raw_targets[index]
    for name, outputs, offset in (("next_token", word_outputs, 0),
                                   ("next_concept", concept_outputs, size),
                                   ("oracle_reconstruction", oracle_reconstructions, 2 * size)):
        text = outputs[index]
        cosine = (float(F.cosine_similarity(generated_vectors[offset + index], reference_vector, dim=0))
                  if text.strip() else 0.0)
        result[name] = {"text": text, "rouge_l_f1": scorer.score(row["target"], text)["rougeL"].fmeasure,
                        "sonar_cosine": cosine}
    rows.append(result)

shared_scores = {}
for name in ("next_token", "next_concept", "oracle_reconstruction"):
    shared_scores[name] = {
        metric: sum(row[name][metric] for row in rows) / len(rows)
        for metric in ("rouge_l_f1", "sonar_cosine")
    }
comparison = {
    "scope": "deterministic_held_out_subset", "split": COMPARISON_SPLIT,
    "examples": len(rows), "pair_ids": [row["pair_id"] for row in rows],
    "pairs_sha256": PAIRS_SHA, "sampling_seed": COMPARISON_SEED,
    "scores": shared_scores, "smoke_test": SMOKE_TEST, "fixture_test": FIXTURE_TEST,
    "live_sonar_used": not FIXTURE_TEST,
    "timing_scope": "cached_context_generation_excluding_model_loading_and_context_embedding",
    "next_token_generation_seconds": word_generation_seconds,
    "concept_predictor_seconds": concept_predictor_seconds,
    "sonar_text_decode_seconds": concept_decode_seconds,
    "word_generation_limit_bpe_tokens": 12 if SMOKE_TEST else WCFG["target_length"],
    "concept_generation_limit_sonar_tokens": SONAR_DECODE_LIMIT,
    "records": rows,
}
(RUN_DIR / "comparison.json").write_text(json.dumps(comparison, ensure_ascii=False, indent=2) + "\n")
print(json.dumps(shared_scores, indent=2))
print("\nCONTEXT:", rows[0]["context"])
print("REFERENCE:", rows[0]["reference"])
print("NEXT TOKEN:", rows[0]["next_token"]["text"])
print("NEXT CONCEPT:", rows[0]["next_concept"]["text"])
print("ORACLE RECONSTRUCTION:", rows[0]["oracle_sonar_reconstruction"])


## 18. Record training cost without mixing losses

The two trainable predictors have different input/output layers, parameter counts and effective sequence lengths.
The concept experiment additionally uses a large frozen pretrained encoder and decoder.
Its cached-vector training time excludes SONAR preprocessing; both costs are reported separately.

For a stronger study, run multiple seeds, match a parameter or compute budget, keep the same evaluation IDs,
and evaluate factual correctness. This notebook does not establish that one objective is superior.


In [ ]:
import csv

word_metrics = json.loads((WORD_RUN_DIR / "metrics.json").read_text())
cost_rows = [
    {"objective": "next_token", "trainable_parameters": word_metrics["parameters"],
     "optimizer_updates": word_metrics["optimizer_updates"],
     "training_and_validation_seconds": word_metrics["training_seconds"],
     "pretrained_sonar_parameters": 0, "initial_sentence_encoding_seconds": 0.0,
     "measured_preprocessing_seconds_this_run": word_metrics["measured_preprocessing_seconds"],
     **shared_scores["next_token"]},
    {"objective": "next_concept", "trainable_parameters": PARAMETERS,
     "optimizer_updates": metrics["optimizer_updates"],
     "training_and_validation_seconds": metrics["training_seconds"],
     "pretrained_sonar_parameters": sum(frozen_parameters.values()),
     "initial_sentence_encoding_seconds": embedding_seconds_original,
     "measured_preprocessing_seconds_this_run": embedding_seconds_this_run + NORMALIZATION_SECONDS,
     **shared_scores["next_concept"]},
]
with (RUN_DIR / "comparison.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(cost_rows[0]))
    writer.writeheader()
    writer.writerows(cost_rows)
metrics.update({"frozen_sonar_parameters": frozen_parameters,
                "model_loading_seconds": model_load_seconds,
                "comparison_scope": comparison["scope"],
                "shared_text_scores": shared_scores,
                "pair_fingerprint": PAIRS_SHA})
(RUN_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
print(json.dumps(cost_rows, indent=2))
if FIXTURE_TEST:
    print("TEST FIXTURES ONLY: all concept/text scores are non-semantic wiring checks, not research results.")


## 19. Optional multi-concept rollout

Each iteration predicts one new sentence vector and adds it to the context.
The newly predicted vector is used directly, not an embedding of a reference sentence.
This extends the one-step objective to autoregressive sentence-level generation.

The demo has a fixed step limit; learned end-of-document stopping is not implemented.
It may accumulate errors and should not be used as a measured benchmark.


In [ ]:
RUN_ROLLOUT = False

@torch.no_grad()
def generate_concepts(context_sentences, steps=3):
    raw_context = encode_texts(context_sentences[-CFG["context_sentences"]:])
    context_vectors = (raw_context - center) / scale
    predictions = []
    for _ in range(steps):
        x = context_vectors[-CFG["context_sentences"]:].unsqueeze(0).to(DEVICE)
        mask = torch.ones(1, x.size(1), dtype=torch.bool, device=DEVICE)
        with amp_context():
            next_vector = model(x, mask).float().cpu()[0]
        predictions.append(next_vector * scale + center)
        context_vectors = torch.cat((context_vectors, next_vector.unsqueeze(0)), dim=0)
    return decode_vectors(torch.stack(predictions))

if RUN_ROLLOUT:
    print(generate_concepts(chosen[0]["context"], steps=3))
    clear_sonar()


## 20. Export the concept predictor

The export includes trained predictor weights, normalization, model-card provenance and measured comparison files.
It references the official SONAR assets and does not redistribute their large pretrained weights.
The text encoder/decoder use SONAR's [non-commercial model license](https://github.com/facebookresearch/SONAR/blob/main/NC_MODEL_LICENSE.md).

Published notebooks have empty outputs. CPU fixture checks validate the surrounding code only;
a live SONAR run and full training must be measured in the compatible runtime before reporting research findings.


In [ ]:
torch.save({"model_state": model.state_dict(), "experiment": experiment,
            "center": center, "scale": scale}, RUN_DIR / "concept_inference.pt")
(RUN_DIR / "sonar_provenance.json").write_text(json.dumps(sonar_provenance, indent=2) + "\n")
with zipfile.ZipFile(RUN_DIR / "next_concept_inference.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for name in ["concept_inference.pt", "normalization.pt", "experiment.json",
                 "sonar_provenance.json", "history.json", "metrics.json",
                 "comparison.json", "comparison.csv", "model_checks.json"]:
        archive.write(RUN_DIR / name, arcname=name)
print("Saved:", RUN_DIR / "next_concept_inference.zip")
